### Gold Layer


In [0]:
# Module update and import

%load_ext autoreload
%autoreload 2

from src.gold import run_gold

run_gold(spark)

In [0]:
#Validate 

gold_count = spark.table(
    "beanalytic_case.gold."
    "interest_inflation_monthly"
).count()

print(f"Gold monthly records: {gold_count}")

In [0]:
#Check the schema

spark.table(
    "beanalytic_case.gold."
    "interest_inflation_monthly"
).printSchema()

In [0]:
#Table visualization

display(
    spark.table(
        "beanalytic_case.gold."
        "interest_inflation_monthly"
    )
    .orderBy("reference_month")
)

In [0]:
#Grain validation

gold_grain_validation = spark.sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT reference_month)
            AS distinct_months,
        MIN(reference_month) AS first_month,
        MAX(reference_month) AS last_month
    FROM beanalytic_case.gold.interest_inflation_monthly
    """
)

display(gold_grain_validation)

In [0]:
#Accumulated validation

accumulated_validation = spark.sql(
    """
    SELECT
        COUNT(*) AS total_months,
        SUM(
            CASE
                WHEN selic_accumulated_12m_pct IS NULL
                THEN 1
                ELSE 0
            END
        ) AS months_without_full_window,
        SUM(
            CASE
                WHEN selic_accumulated_12m_pct IS NOT NULL
                THEN 1
                ELSE 0
            END
        ) AS months_with_full_window
    FROM beanalytic_case.gold.interest_inflation_monthly
    """
)

display(accumulated_validation)

In [0]:
run_gold(spark)

In [0]:
gold_count_after_second_run = spark.table(
    "beanalytic_case.gold."
    "interest_inflation_monthly"
).count()

print(
    "Gold records after second run: "
    f"{gold_count_after_second_run}"
)

In [0]:
#Pipeline validation

pipeline_validation = spark.sql(
    """
    SELECT
        'bronze_selic' AS table_name,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT data_raw) AS distinct_keys
    FROM beanalytic_case.bronze.selic

    UNION ALL

    SELECT
        'bronze_ipca',
        COUNT(*),
        COUNT(DISTINCT data_raw)
    FROM beanalytic_case.bronze.ipca

    UNION ALL

    SELECT
        'silver_selic',
        COUNT(*),
        COUNT(DISTINCT reference_date)
    FROM beanalytic_case.silver.selic

    UNION ALL

    SELECT
        'silver_ipca',
        COUNT(*),
        COUNT(DISTINCT reference_month)
    FROM beanalytic_case.silver.ipca

    UNION ALL

    SELECT
        'gold_monthly',
        COUNT(*),
        COUNT(DISTINCT reference_month)
    FROM beanalytic_case.gold.interest_inflation_monthly
    """
)

display(pipeline_validation)

In [0]:
display(
    spark.table(
        "beanalytic_case.gold.interest_inflation_monthly"
    )
    .orderBy("reference_month", ascending=False)
    .limit(12)
)